In [ ]:
!pip install torch torchvision scikit-learn xgboost tqdm joblib pillow matplotlib --quiet

In [ ]:
import zipfile, os

zip_path = '/content/archive (3).zip'   # <-- your uploaded ZIP file
extract_path = '/content/Data'

os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("✅ Extracted successfully to:", extract_path)
!ls -R "$extract_path"

In [ ]:
import os
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from tqdm import tqdm
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from xgboost import XGBClassifier
import joblib
from PIL import Image

# -------------------- PARAMETERS --------------------
data_dir = '/content/Data'
epochs = 8
batch_size = 8
lr = 1e-4
image_size = 224
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
save_dir = '/content/output'
os.makedirs(save_dir, exist_ok=True)

# -------------------- TRANSFORMS --------------------
train_tfms = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])
val_tfms = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

# -------------------- LOAD DATA --------------------
train_path = os.path.join(data_dir, 'train')
valid_path = os.path.join(data_dir, 'valid')
test_path  = os.path.join(data_dir, 'test')

train_ds = datasets.ImageFolder(train_path, transform=train_tfms)
valid_ds = datasets.ImageFolder(valid_path, transform=val_tfms)
test_ds  = datasets.ImageFolder(test_path,  transform=val_tfms)

classes = train_ds.classes
num_classes = len(classes)
print("📁 Classes found:", classes)

train_dl = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=2)
valid_dl = DataLoader(valid_ds, batch_size=batch_size, shuffle=False, num_workers=2)
test_dl  = DataLoader(test_ds,  batch_size=batch_size, shuffle=False, num_workers=2)

# -------------------- MODEL: RESNET50 --------------------
model = models.resnet50(pretrained=True)
for param in model.parameters():
    param.requires_grad = True
in_features = model.fc.in_features
model.fc = nn.Linear(in_features, num_classes)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=lr)

# -------------------- TRAIN --------------------
best_acc = 0
for epoch in range(epochs):
    model.train()
    train_loss, train_correct, total = 0, 0, 0
    for x, y in tqdm(train_dl, desc=f"Epoch {epoch+1}/{epochs}"):
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        out = model(x)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * x.size(0)
        train_correct += (out.argmax(1) == y).sum().item()
        total += y.size(0)
    train_acc = train_correct / total

    # validation
    model.eval()
    val_correct, total = 0, 0
    with torch.no_grad():
        for x, y in valid_dl:
            x, y = x.to(device), y.to(device)
            out = model(x)
            val_correct += (out.argmax(1) == y).sum().item()
            total += y.size(0)
    val_acc = val_correct / total

    print(f"Epoch [{epoch+1}/{epochs}] - Train Acc: {train_acc:.4f}, Val Acc: {val_acc:.4f}")
    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), f"{save_dir}/best_resnet50.pth")
        print("✅ Best model saved.")

# -------------------- FEATURE EXTRACTION --------------------
def extract_features(model, loader):
    feats, labels = [], []
    feature_model = torch.nn.Sequential(*(list(model.children())[:-1]))  # remove final FC
    feature_model.eval().to(device)
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            f = feature_model(x)
            f = torch.flatten(f, 1)
            feats.append(f.cpu().numpy())
            labels.append(y.numpy())
    return np.concatenate(feats), np.concatenate(labels)

print("\n📊 Extracting features for XGBoost...")
train_feats, train_labels = extract_features(model, train_dl)
valid_feats, valid_labels = extract_features(model, valid_dl)
test_feats,  test_labels  = extract_features(model, test_dl)

# -------------------- XGBOOST CLASSIFIER --------------------
print("🌲 Training XGBoost classifier...")
clf = XGBClassifier(use_label_encoder=False, eval_metric='mlogloss', verbosity=0)
clf.fit(train_feats, train_labels)
joblib.dump(clf, f"{save_dir}/xgb.pkl")

# -------------------- EVALUATION --------------------
def get_probs(model, loader):
    model.eval()
    all_probs, all_labels = [], []
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            out = torch.softmax(model(x), dim=1).cpu().numpy()
            all_probs.append(out)
            all_labels.append(y.numpy())
    return np.concatenate(all_probs), np.concatenate(all_labels)

probs_cnn, labels = get_probs(model, test_dl)
probs_xgb = clf.predict_proba(test_feats)
ensemble_probs = (probs_cnn + probs_xgb) / 2
ensemble_preds = ensemble_probs.argmax(1)

cnn_acc = accuracy_score(labels, probs_cnn.argmax(1))
xgb_acc = accuracy_score(labels, probs_xgb.argmax(1))
ens_acc = accuracy_score(labels, ensemble_preds)

print(f"\n📈 CNN Accuracy: {cnn_acc:.4f}")
print(f"🌲 XGBoost Accuracy: {xgb_acc:.4f}")
print(f"🤝 Ensemble Accuracy: {ens_acc:.4f}\n")

print("Classification Report:\n", classification_report(labels, ensemble_preds, target_names=classes))
print("Confusion Matrix:\n", confusion_matrix(labels, ensemble_preds))

In [ ]:
from PIL import Image

def predict_image(img_path):
    img = Image.open(img_path).convert('RGB')
    x = val_tfms(img).unsqueeze(0).to(device)
    with torch.no_grad():
        out_cnn = torch.softmax(model(x), dim=1).cpu().numpy()[0]
        # Feature + XGB
        feature_model = torch.nn.Sequential(*(list(model.children())[:-1])).to(device)
        f = feature_model(x)
        f = torch.flatten(f, 1).cpu().numpy()
        out_xgb = clf.predict_proba(f)[0]
    probs = (out_cnn + out_xgb) / 2
    pred = classes[np.argmax(probs)]
    print(f"🩻 Predicted Class: {pred}")
    print({classes[i]: round(float(p),4) for i,p in enumerate(probs)})

# Example usage:
# predict_image('/content/Data/test/normal/image_1.png')

In [3]:
import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image
import joblib
import os

# Device setup
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# --- 1. Load the trained CNN model ---
model = models.resnet50(pretrained=False)
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, 4)   # 4 classes: adjust if different
model.load_state_dict(torch.load('C:/Users/srkap/Downloads/output_files/content/output/best_resnet50.pth', map_location=device))
model = model.to(device)
model.eval()

# --- 2. Load the XGBoost model ---
xgb_model = joblib.load('xgb.pkl')

# --- 3. Define image transforms ---
val_tfms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

# --- 4. Create feature extractor (remove the final layer) ---
feature_model = nn.Sequential(*list(model.children())[:-1]).to(device)
feature_model.eval()

# --- 5. Prediction function ---
def predict_image(img_path):
    img = Image.open(img_path).convert('RGB')
    x = val_tfms(img).unsqueeze(0).to(device)

    with torch.no_grad():
        features = feature_model(x)
        features = features.view(features.size(0), -1).cpu().numpy()

    preds = xgb_model.predict(features)
    classes = ['adenocarcinoma', 'large.cell.carcinoma', 'normal', 'squamous.cell.carcinoma']
    return classes[int(preds[0])]

# --- 6. Predict on all test images ---
test_dir = r'C:/Users/srkap/Downloads/output_files/content/output/archive (3)/Data/test'

for cls in os.listdir(test_dir):
    cls_path = os.path.join(test_dir, cls)
    if os.path.isdir(cls_path):
        for img_name in os.listdir(cls_path):
            img_path = os.path.join(cls_path, img_name)
            pred = predict_image(img_path)
            print(f"{img_name} → Predicted: {pred} | True: {cls}")


000108 (3).png → Predicted: adenocarcinoma | True: adenocarcinoma
000109 (2).png → Predicted: adenocarcinoma | True: adenocarcinoma
000109 (4).png → Predicted: adenocarcinoma | True: adenocarcinoma
000109 (5).png → Predicted: adenocarcinoma | True: adenocarcinoma
000112 (2).png → Predicted: adenocarcinoma | True: adenocarcinoma
000113 (7).png → Predicted: adenocarcinoma | True: adenocarcinoma
000114 (5).png → Predicted: adenocarcinoma | True: adenocarcinoma
000114.png → Predicted: adenocarcinoma | True: adenocarcinoma
000115 (4).png → Predicted: large.cell.carcinoma | True: adenocarcinoma
000115 (8).png → Predicted: adenocarcinoma | True: adenocarcinoma
000115.png → Predicted: large.cell.carcinoma | True: adenocarcinoma
000116 (5).png → Predicted: adenocarcinoma | True: adenocarcinoma
000116 (7).png → Predicted: adenocarcinoma | True: adenocarcinoma
000116 (9).png → Predicted: adenocarcinoma | True: adenocarcinoma
000117 (4).png → Predicted: adenocarcinoma | True: adenocarcinoma
000117

In [5]:
print("\n🔹 Predicting a single image...")
single_image_path = r"C:/Users/srkap/Downloads/output_files/content/output/archive (3)/Data/test/normal/6 - Copy (2) - Copy.png"  # 👈 change path
pred_class = predict_image(single_image_path)
print(f"Single image prediction → {os.path.basename(single_image_path)}: {pred_class}")


🔹 Predicting a single image...
Single image prediction → 6 - Copy (2) - Copy.png: normal
